[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/01_finetuning_and_architecture/03_metric_vault_extractor.ipynb)

# Armory Notebook 3 — Metric Vault Extractor

| | |
|---|---|
| **Companion to** | Lesson 3 — Boss Fight prerequisites (the Metric Vault in PROGRESS.md) |
| **Runtime** | CPU; needs *your* prediction/embedding artifacts for REAL mode |
| **Estimated time** | 25 minutes |
| **XP** | **0.** This notebook mines the numbers; the boss fight is where they earn XP. |
| **Last verified** | 2026-07-13 |

The boss fight assumes a filled Metric Vault. This notebook computes every vault row that can be derived from saved artifacts: per-class F1, macro/micro-F1, the confusion structure, and the embedding-space diagnostics (alignment/uniformity) for the contrastive project.

> [!IMPORTANT]
> **Two modes.** With `PREDICTIONS_CSV = None` the notebook runs on clearly-labeled **synthetic demo data** so you can see the machinery work. Demo numbers must never enter the Metric Vault, your pitch, or your resume — the game never pays for a made-up number. Point the paths at your real ESCI prediction dump and QQP embeddings to switch to REAL mode.

In [ ]:
%pip install -q "numpy>=1.26" "scikit-learn>=1.4" "matplotlib>=3.8" "pandas>=2.1"

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- POINT THESE AT YOUR ARTIFACTS FOR REAL MODE -------------------------
PREDICTIONS_CSV = None   # e.g. "runs/qlora_esci/test_predictions.csv" with columns: true_label, pred_label
PAIR_EMBEDDINGS_NPZ = None  # e.g. "runs/rope_qqp/eval_embeddings.npz" with arrays: anchor, positive
# ---------------------------------------------------------------------------

ESCI_CLASSES = ["Exact", "Substitute", "Complement", "Irrelevant"]
rng = np.random.default_rng(11)

if PREDICTIONS_CSV is None:
    DEMO_MODE = True
    # Synthetic, imbalance-shaped stand-in: exists only so the plots render.
    class_mix = [0.62, 0.21, 0.05, 0.12]
    true_labels = rng.choice(4, size=3000, p=class_mix)
    flip_to = {0: [1], 1: [0, 2], 2: [1, 3], 3: [2, 0]}
    pred_labels = true_labels.copy()
    noisy = rng.random(3000) < np.take([0.12, 0.35, 0.55, 0.25], true_labels)
    pred_labels[noisy] = [rng.choice(flip_to[t]) for t in true_labels[noisy]]
    print("MODE: SYNTHETIC DEMO - numbers below are fabricated shapes, not results.")
else:
    DEMO_MODE = False
    frame = pd.read_csv(PREDICTIONS_CSV)
    missing = {"true_label", "pred_label"} - set(frame.columns)
    if missing:
        raise ValueError(f"{PREDICTIONS_CSV} is missing columns: {missing}")
    true_labels = frame["true_label"].to_numpy()
    pred_labels = frame["pred_label"].to_numpy()
    print(f"MODE: REAL - loaded {len(frame):,} predictions from {PREDICTIONS_CSV}")

DEMO_TAG = "SYNTHETIC DEMO — NOT YOUR RESULTS" if DEMO_MODE else ""

## Vault rows 1–3: macro-F1, per-class F1, and what the average hides

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score

per_class_f1 = f1_score(true_labels, pred_labels, average=None, labels=range(4))
print(f"{DEMO_TAG}\n" if DEMO_MODE else "", end="")
print(f"macro-F1 : {f1_score(true_labels, pred_labels, average='macro'):.4f}")
print(f"micro-F1 : {f1_score(true_labels, pred_labels, average='micro'):.4f}")
print(f"accuracy : {accuracy_score(true_labels, pred_labels):.4f}\n")
for name, score in zip(ESCI_CLASSES, per_class_f1):
    support = int((true_labels == ESCI_CLASSES.index(name)).sum())
    print(f"  F1[{name:<10}] = {score:.4f}   (support {support:,})")
print("\nBoss-fight framing: the macro/micro gap plus the weakest per-class F1")
print("IS the answer to 'what does your headline metric hide?'")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

INK, INK_SOFT, SERIES_BLUE, SERIES_AQUA = "#0b0b0b", "#52514e", "#2a78d6", "#1baf7a"
blue_ramp = LinearSegmentedColormap.from_list(
    "vault_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)

conf = confusion_matrix(true_labels, pred_labels, labels=range(4))
row_norm = conf / conf.sum(axis=1, keepdims=True)

fig, (ax_cm, ax_f1) = plt.subplots(1, 2, figsize=(11, 4.2))
if DEMO_MODE:
    fig.suptitle(DEMO_TAG, color="#e34948", fontsize=10, y=1.02)

heat = ax_cm.imshow(row_norm, cmap=blue_ramp, vmin=0, vmax=1)
ax_cm.set_xticks(range(4), ESCI_CLASSES, fontsize=8, color=INK)
ax_cm.set_yticks(range(4), ESCI_CLASSES, fontsize=8, color=INK)
ax_cm.set_xlabel("Predicted", color=INK_SOFT, fontsize=9)
ax_cm.set_ylabel("True", color=INK_SOFT, fontsize=9)
ax_cm.set_title("Row-normalized confusion", fontsize=10, color=INK)
for i in range(4):
    for j in range(4):
        label_ink = "#ffffff" if row_norm[i, j] > 0.55 else INK
        ax_cm.text(j, i, f"{conf[i, j]:,}", ha="center", va="center",
                   fontsize=8, color=label_ink)
fig.colorbar(heat, ax=ax_cm, shrink=0.85)

bars = ax_f1.bar(ESCI_CLASSES, per_class_f1, color=SERIES_BLUE, width=0.62)
ax_f1.bar_label(bars, fmt="%.3f", fontsize=8, color=INK, padding=2)
ax_f1.set_ylim(0, 1.05)
ax_f1.set_title("Per-class F1", fontsize=10, color=INK)
ax_f1.tick_params(colors=INK_SOFT, labelsize=8)
ax_f1.spines[["top", "right"]].set_visible(False)
ax_f1.grid(axis="y", color="#e6e5e0", linewidth=0.7)
ax_f1.set_axisbelow(True)

plt.tight_layout()
plt.show()
print("Read the off-diagonal: which boundary dominates - S/C? E/S? That sentence is")
print("your answer to boss-fight Follow-up 2, and it must match this matrix exactly.")

## Vault rows: contrastive embedding diagnostics (alignment & uniformity)

Wang & Isola (2020), *Understanding Contrastive Representation Learning through Alignment and Uniformity on the Hypersphere* (https://arxiv.org/abs/2005.10242): **alignment** = mean squared distance between positive pairs (lower = paraphrases embed together); **uniformity** = log of the mean Gaussian potential over random pairs (lower = embeddings spread over the sphere). Collapse shows up as excellent alignment with terrible uniformity.

In [ ]:
if PAIR_EMBEDDINGS_NPZ is None:
    emb_demo = True
    latent = rng.normal(size=(1500, 64))
    anchor_vecs = latent + 0.35 * rng.normal(size=latent.shape)      # paraphrase pairs share
    positive_vecs = latent + 0.35 * rng.normal(size=latent.shape)    # a latent 'meaning'
    print("MODE: SYNTHETIC DEMO embeddings - shapes only, not your encoder.")
else:
    emb_demo = False
    packed = np.load(PAIR_EMBEDDINGS_NPZ)
    anchor_vecs, positive_vecs = packed["anchor"], packed["positive"]
    print(f"MODE: REAL - {anchor_vecs.shape[0]:,} pairs, dim {anchor_vecs.shape[1]}")

def unit(rows: np.ndarray) -> np.ndarray:
    return rows / np.linalg.norm(rows, axis=1, keepdims=True)

za, zp = unit(anchor_vecs), unit(positive_vecs)
alignment = float((np.linalg.norm(za - zp, axis=1) ** 2).mean())

shuffle = rng.permutation(len(za))
sq_dists = (np.linalg.norm(za - za[shuffle], axis=1) ** 2)
uniformity = float(np.log(np.exp(-2.0 * sq_dists).mean()))

pos_cosine = (za * zp).sum(axis=1)
rand_cosine = (za * za[shuffle]).sum(axis=1)

print(f"\nalignment  (lower is tighter pairs) : {alignment:.4f}")
print(f"uniformity (lower is better spread) : {uniformity:.4f}")
print(f"mean cosine  positives vs random    : {pos_cosine.mean():.3f} vs {rand_cosine.mean():.3f}")
print("\nCollapse check: if both distributions in the next plot sit on top of each")
print("other near cosine 1.0, the encoder collapsed - Lesson 2, Follow-up 6.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.6))
if emb_demo:
    fig.suptitle("SYNTHETIC DEMO — NOT YOUR RESULTS", color="#e34948", fontsize=10, y=1.03)

shared_bins = np.linspace(-0.4, 1.0, 57)
ax.hist(pos_cosine, bins=shared_bins, color=SERIES_BLUE, alpha=0.75,
        label="Labeled duplicate pairs", edgecolor="#fcfcfb", linewidth=0.4)
ax.hist(rand_cosine, bins=shared_bins, color=SERIES_AQUA, alpha=0.75,
        label="Random pairs", edgecolor="#fcfcfb", linewidth=0.4)
ax.set_xlabel("Cosine similarity", color=INK_SOFT, fontsize=9)
ax.set_ylabel("Pair count", color=INK_SOFT, fontsize=9)
ax.set_title("Separation of duplicate vs. random pairs", fontsize=10, color=INK)
ax.legend(fontsize=8, frameon=False)
ax.tick_params(colors=INK_SOFT, labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e6e5e0", linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()
print("The overlap region between the two histograms is your error mass - and the")
print("place hard-negative mining would act. Describing THIS picture from memory is")
print("what 'I studied my embedding space' means in the room.")

## Recording protocol

**REAL mode only.** Copy into the Metric Vault in [PROGRESS.md](../../PROGRESS.md):

| Vault row | Cell that produced it |
|---|---|
| QLoRA: macro-F1 (test) | Vault rows 1–3 cell |
| QLoRA: per-class F1 (E/S/C/I) | Vault rows 1–3 cell |
| ROPE model: QQP eval metric + value | Alignment/uniformity cell (plus your task metric from your eval script) |
| Contrastive diagnostics (alignment, uniformity, cosine separation) | Both embedding cells |

For each value, also record the **source path** column — the artifact file this notebook read. That column is your run-log traceability, and it's what lets you say "I can show you the log" in the room.

**DEMO mode:** record nothing. If you ran demo mode because the artifacts don't exist, that discovery *is* the finding: your projects currently make claims whose evidence files you can't produce. Regenerating them (re-running eval with a predictions dump) is tonight's highest-value hour.

## References & Credits

- Wang & Isola (2020), *Understanding Contrastive Representation Learning through Alignment and Uniformity on the Hypersphere* — https://arxiv.org/abs/2005.10242 (alignment/uniformity definitions implemented above).
- scikit-learn (BSD-3), NumPy (BSD-3), Matplotlib (PSF-based license), pandas (BSD-3).